# 02 - Feature Engineering

Xây dựng và tối ưu hóa các đặc trưng cho mô hình dự đoán giá nhà

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

sys.path.insert(0, os.path.abspath('../src'))

from data.load_data import load_raw_data, save_data
from data.preprocess import handle_missing_values, remove_outliers, encode_categorical, scale_features
from features.build_features import create_polynomial_features, create_interaction_features, select_features
from utils.helpers import print_data_info

## 2. Load Data

In [ ]:
# Load data
raw_data_path = '../data/raw/house_prices.csv'
df = load_raw_data(raw_data_path)
print(f"Original data shape: {df.shape}")

## 3. Handle Missing Values

In [ ]:
# Handle missing values
df = handle_missing_values(df, strategy='mean')

## 4. Remove Outliers

In [ ]:
# Get numeric columns for outlier removal
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Remove outliers using Z-score
if 'price' in numeric_cols:  # Assuming 'price' is the target variable
    numeric_cols_for_outlier = [col for col in numeric_cols if col != 'price']
else:
    numeric_cols_for_outlier = numeric_cols

df = remove_outliers(df, numeric_cols_for_outlier, threshold=3.0)

## 5. Encode Categorical Variables

In [ ]:
# Get categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns: {categorical_cols}")

# Encode categorical variables
if categorical_cols:
    df = encode_categorical(df, categorical_cols, method='label')

## 6. Create Polynomial Features

In [ ]:
# Create polynomial features for selected numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
selected_cols = numeric_cols[:3] if len(numeric_cols) > 3 else numeric_cols

df = create_polynomial_features(df, selected_cols, degree=2)
print(f"Data shape after polynomial features: {df.shape}")

## 7. Feature Selection

In [ ]:
# Select features using correlation with target
# Assuming 'price' is the target variable
if 'price' in df.columns:
    selected_features = select_features(df, target='price', method='correlation', k=10)
    print(f"\nSelected features: {selected_features}")
else:
    print("Target variable 'price' not found in data")

## 8. Scale Features

In [ ]:
# Identify numeric columns for scaling
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Scale features
df_scaled, scaler = scale_features(df, columns=numeric_cols, scaler_type='standard')
print(f"Data scaled successfully")

## 9. Save Processed Data

In [ ]:
# Save processed data
processed_data_path = '../data/processed/house_prices_processed.csv'
save_data(df_scaled, processed_data_path)
print(f"Processed data shape: {df_scaled.shape}")
print("\nProcessed data preview:")
print(df_scaled.head())